# Etapa 4B: comparação entre a pipeline clássica e o detector YOLO, e demonstração

Antes de rodar, confira no Drive, em `MyDrive/PDI_Contagem_Objetos/etapa4/`:

| Arquivo | Situação |
|---|---|
| `saidas/` | já criada pela etapa 4 (não mexer) |
| `data_final.zip` | já está lá |
| `classico/previsoes_test.csv` | **subir**: cópia de `resultados/classico/previsoes_test.csv` do repositório |
| `classico/metricas_test.csv` | opcional: cópia de `resultados/avaliacao/classico_test/metricas.csv`, para conferir as métricas da pipeline clássica |
| `auditoria/exclusoes_teste.csv` | opcional: lista de exclusões da auditoria |

Este notebook só lê os resultados congelados. Ele não altera previsões, limiares nem o test.

- Comparação completa: rode as células 1 a 8, em ordem.
- Só a demonstração com uma imagem qualquer: rode a célula 1 e depois a 9.


### Célula 1 — Ambiente, arquivos e modelos

Monta o Drive, instala o `ultralytics` em versão fixa e confere que os arquivos necessários existem e que `config_yolo_final.json` tem o SHA-256 congelado na etapa 4. Carrega os três detectores e define as funções `detectar` e `desenhar`, usadas nas células seguintes.


In [ ]:
# Célula 1 — Ambiente, arquivos e modelos
import io, sys, hashlib, zipfile, subprocess
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.219"], check=True)

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from ultralytics import YOLO

ETAPA4 = Path("/content/drive/MyDrive/PDI_Contagem_Objetos/etapa4")
SAIDAS = ETAPA4 / "saidas"
CLASSICO_DIR = ETAPA4 / "classico"
AUDITORIA_DIR = ETAPA4 / "auditoria"
COMPARACAO = SAIDAS / "comparacao"

CLASSES = ["parafuso", "porca", "arruela"]
LIMIARES = {"parafuso": 0.30, "porca": 0.15, "arruela": 0.80}  # congelados na etapa 4 (célula 7)
SHA_CONFIG = "2e4858358bb3c04ceeb409e75ac3c0e819d88cf28d62c134dfab7fec22df590d"
CORES_BGR = {"parafuso": (0, 140, 255), "porca": (255, 110, 30), "arruela": (60, 170, 0)}
COR_GT_BGR = (255, 0, 255)  # magenta
DEVICE = 0 if torch.cuda.is_available() else "cpu"


def sha256(caminho):
    """SHA-256 de um arquivo, lido em blocos de 1 MiB."""
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(1 << 20), b""):
            h.update(bloco)
    return h.hexdigest()


pesos = {c: SAIDAS / "pesos" / f"{c}_best.pt" for c in CLASSES}
necessarios = [SAIDAS / "config_yolo_final.json", ETAPA4 / "data_final.zip", *pesos.values()]
faltando = [str(p) for p in necessarios if not p.exists()]
assert not faltando, "Arquivos faltando no Drive:\n" + "\n".join(faltando)
assert sha256(SAIDAS / "config_yolo_final.json") == SHA_CONFIG, "config_yolo_final.json não é o congelado na etapa 4."

DATA = Path("/content/data_final")
if not (DATA / "contagens_gt.csv").exists():
    destino = Path("/content/_data_final_zip")
    with zipfile.ZipFile(ETAPA4 / "data_final.zip") as z:
        z.extractall(destino)
    achados = list(destino.rglob("contagens_gt.csv"))
    assert len(achados) == 1, f"Esperado 1 contagens_gt.csv no zip; encontrado(s): {achados}"
    DATA = achados[0].parent

modelos = {c: YOLO(str(p)) for c, p in pesos.items()}
COMPARACAO.mkdir(parents=True, exist_ok=True)


def detectar(modelo, imagem, limiar):
    """Mesmo critério da etapa 4: inferência com confiança mínima 0,05 e contagem das caixas com confiança >= limiar."""
    r = modelo.predict(imagem, conf=0.05, imgsz=640, max_det=300, device=DEVICE, verbose=False)[0]
    caixas = r.boxes.xyxy.cpu().numpy()
    confs = r.boxes.conf.cpu().numpy()
    manter = confs >= limiar
    return caixas[manter], confs[manter]


def desenhar(bgr, caixas, cor, rotulos=None, espessura=None):
    """Desenha as caixas (e os rótulos, se houver) sobre a imagem BGR e a devolve."""
    esp = espessura or max(1, round(max(bgr.shape[:2]) / 400))
    for i, (x1, y1, x2, y2) in enumerate(np.asarray(caixas).astype(int)):
        cv2.rectangle(bgr, (x1, y1), (x2, y2), cor, esp)
        if rotulos is not None:
            cv2.putText(bgr, rotulos[i], (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX,
                        0.4 * esp, cor, max(1, esp - 1), cv2.LINE_AA)
    return bgr


print(f"Tudo conferido. Dados em {DATA} | dispositivo: {DEVICE}")

### Célula 2 — Previsões dos dois métodos casadas com o GT

Junta as previsões da pipeline clássica e do detector YOLO com o GT do test e falha se alguma linha não casar. Para aplicar a auditoria, mude `USAR_EXCLUSOES` para `True` e rode de novo da célula 2 em diante. Os arquivos gerados passam a ter o sufixo `com_exclusoes`.


In [ ]:
# Célula 2 — Previsões dos dois métodos casadas com o GT (test)
USAR_EXCLUSOES = False  # True para aplicar auditoria/exclusoes_teste.csv

CHAVE = ["dataset", "imagem", "classe"]
gt = pd.read_csv(DATA / "contagens_gt.csv")
gt = gt[gt["split"] == "test"][CHAVE + ["contagem"]].rename(columns={"contagem": "real"})


def carregar_previsoes(caminho):
    """Lê um CSV de previsões, fica só com o test e devolve (nome do método, contagens por imagem e classe)."""
    assert caminho.exists(), f"Arquivo não encontrado: {caminho}"
    p = pd.read_csv(caminho)
    p = p[p["split"] == "test"]
    metodos = p["metodo"].unique().tolist()
    assert len(metodos) == 1, f"{caminho}: esperado 1 método, encontrado(s) {metodos}"
    assert not p.duplicated(CHAVE).any(), f"{caminho}: linhas duplicadas"
    return metodos[0], p[CHAVE + ["contagem"]]


M_CLS, prev_cls = carregar_previsoes(CLASSICO_DIR / "previsoes_test.csv")
M_YOLO, prev_yolo = carregar_previsoes(SAIDAS / "previsoes_test.csv")
METODOS = [M_CLS, M_YOLO]

df = gt.copy()
for nome, prev in [(M_CLS, prev_cls), (M_YOLO, prev_yolo)]:
    m = df.merge(prev, on=CHAVE, how="outer", indicator=True)
    problema = m[m["_merge"] != "both"]
    assert problema.empty, f"{nome}: previsões não casam com o GT do test:\n{problema}"
    df = m.drop(columns="_merge").rename(columns={"contagem": nome})

if USAR_EXCLUSOES:
    arq = AUDITORIA_DIR / "exclusoes_teste.csv"
    assert arq.exists(), f"Exclusões não encontradas: {arq}"
    excl = pd.read_csv(arq)
    chaves = set(zip(excl["dataset"], excl["imagem"]))
    inexistentes = chaves - set(zip(df["dataset"], df["imagem"]))
    assert not inexistentes, f"Exclusões que não existem no test: {inexistentes}"
    df = df[[(d, i) not in chaves for d, i in zip(df["dataset"], df["imagem"])]].reset_index(drop=True)
    print(f"Exclusões aplicadas: {len(chaves)} imagem(ns).")

SUFIXO = "com_exclusoes" if USAR_EXCLUSOES else "sem_exclusoes"
n_imgs = df[["dataset", "imagem"]].drop_duplicates().shape[0]
print(f"Métodos: {M_CLS} × {M_YOLO} | {n_imgs} imagens | {len(df)} linhas imagem×classe")

### Célula 3 — Métricas por escopo e tabela comparativa

Calcula MAE, RMSE, viés, acerto exato e acerto ±1 de cada método, no geral, por classe e por classe e dataset, e monta a tabela comparativa com a redução do MAE. Sem exclusões, confere os valores contra os `metricas.csv` do `avaliar.py`.


In [ ]:
# Célula 3 — Métricas por escopo e tabela comparativa
def metricas(g, metodo):
    """MAE, RMSE, viés, acerto exato e acerto ±1 do método sobre as linhas de g."""
    erro = g[metodo] - g["real"]
    return {
        "n": len(g),
        "soma_real": int(g["real"].sum()),
        "soma_prevista": int(g[metodo].sum()),
        "mae": erro.abs().mean(),
        "rmse": float(np.sqrt((erro ** 2).mean())),
        "vies": erro.mean(),
        "exato_pct": (erro == 0).mean() * 100,
        "tol1_pct": (erro.abs() <= 1).mean() * 100,
    }


grupos = [("geral", "TODAS", "TODOS", df)]
grupos += [("classe", c, "TODOS", g) for c, g in df.groupby("classe")]
grupos += [("classe_dataset", c, d, g) for (c, d), g in df.groupby(["classe", "dataset"])]
tab = pd.DataFrame([{"escopo": e, "classe": c, "dataset": d, "metodo": m, **metricas(g, m)}
                    for e, c, d, g in grupos for m in METODOS])

IDX = ["escopo", "classe", "dataset"]
a = tab[tab["metodo"] == M_CLS].set_index(IDX)
b = tab[tab["metodo"] == M_YOLO].set_index(IDX)
comparacao = pd.DataFrame({
    "n": a["n"],
    "MAE clássico": a["mae"], "MAE YOLO": b["mae"],
    "redução do MAE (%)": (a["mae"] - b["mae"]) / a["mae"] * 100,
    "viés clássico": a["vies"], "viés YOLO": b["vies"],
    "exato clássico (%)": a["exato_pct"], "exato YOLO (%)": b["exato_pct"],
    "±1 clássico (%)": a["tol1_pct"], "±1 YOLO (%)": b["tol1_pct"],
}).round(2)
display(comparacao)
comparacao.to_csv(COMPARACAO / f"comparacao_metricas_{SUFIXO}.csv")


def conferir(arquivo, metodo):
    """Confere as métricas calculadas aqui com as do avaliar.py (só sem exclusões)."""
    if not arquivo.exists():
        print(f"(sem {arquivo} para conferir {metodo})")
        return
    nomes = {"mae": "mae", "vies": "vies", "acerto_exato_pct": "exato_pct", "acerto_tol1_pct": "tol1_pct"}
    ref = pd.read_csv(arquivo)
    ref = ref[(ref["metodo"] == metodo) & ref["metrica"].isin(list(nomes))]
    assert not ref.empty, f"{arquivo}: nenhuma métrica de {metodo} encontrada"
    nossa = tab[tab["metodo"] == metodo].set_index(IDX)
    for _, r in ref.iterrows():
        v = nossa.loc[(r["escopo"], r["classe"], r["dataset"]), nomes[r["metrica"]]]
        assert abs(v - r["valor"]) < 1e-3, (
            f"{metodo} diverge do avaliar.py em {r['escopo']}/{r['classe']}/{r['dataset']}/{r['metrica']}: "
            f"{v} × {r['valor']}")
    print(f"{metodo}: {len(ref)} valores idênticos aos do avaliar.py")


if not USAR_EXCLUSOES:
    conferir(SAIDAS / "avaliacao" / f"{M_YOLO}_test" / "metricas.csv", M_YOLO)
    conferir(CLASSICO_DIR / "metricas_test.csv", M_CLS)

### Célula 4 — Erro na contagem total por imagem

Soma as classes de cada imagem e mede o erro absoluto médio da contagem total por dataset, também como proporção do número médio de peças. O formato é o mesmo da tabela da Etapa 3.


In [ ]:
# Célula 4 — Erro na contagem total por imagem (mesmo formato da tabela da Etapa 3)
tot = df.groupby(["dataset", "imagem"])[["real", *METODOS]].sum().reset_index()
linhas = []
for d, g in tot.groupby("dataset"):
    pecas = g["real"].mean()
    linha = {"dataset": d, "imagens": len(g), "peças por imagem": pecas}
    for m, rotulo in [(M_CLS, "clássico"), (M_YOLO, "YOLO")]:
        erro_medio = (g[m] - g["real"]).abs().mean()
        linha[f"erro médio {rotulo}"] = erro_medio
        linha[f"erro relativo {rotulo} (%)"] = erro_medio / pecas * 100
    linhas.append(linha)
total_por_imagem = pd.DataFrame(linhas).round(2)
display(total_por_imagem)
total_por_imagem.to_csv(COMPARACAO / f"total_por_imagem_{SUFIXO}.csv", index=False)

### Célula 5 — Análise pareada e teste de Wilcoxon

Compara, para cada imagem e classe, qual método erra menos e aplica o teste de Wilcoxon sobre os erros absolutos. O teste descarta os empates e tem pouco poder quando há poucas imagens, como em `porca`.


In [ ]:
# Célula 5 — Análise pareada e teste de Wilcoxon (erros absolutos por imagem×classe)
from scipy.stats import wilcoxon

linhas = []
for classe, g in [("TODAS", df), *df.groupby("classe")]:
    e_cls = (g[M_CLS] - g["real"]).abs()
    e_yolo = (g[M_YOLO] - g["real"]).abs()
    dif = e_cls - e_yolo
    linhas.append({
        "classe": classe,
        "n": len(g),
        "YOLO erra menos": int((dif > 0).sum()),
        "empate": int((dif == 0).sum()),
        "clássico erra menos": int((dif < 0).sum()),
        "diferença média de erro (clássico − YOLO)": dif.mean(),
        "p-valor (Wilcoxon)": wilcoxon(e_cls, e_yolo).pvalue if (dif != 0).any() else np.nan,
    })
pareada = pd.DataFrame(linhas).round(4)
display(pareada)
pareada.to_csv(COMPARACAO / f"pareada_wilcoxon_{SUFIXO}.csv", index=False)
print("Obs.: o Wilcoxon descarta os empates; com n pequeno (porca), o teste tem pouco poder.")

### Célula 6 — Hipóteses levantadas na Etapa 3

Examina dois recortes que a Etapa 3 apontou como difíceis para a pipeline clássica: as 10 piores imagens da pipeline clássica em `washers_hardee` e as porcas de `screws_and_nuts`. Mostra o que muda com o YOLO nesses recortes.


In [ ]:
# Célula 6 — Hipóteses levantadas na Etapa 3
w = df[df["dataset"] == "washers_hardee"].copy()
w["erro clássico"] = (w[M_CLS] - w["real"]).abs()
w["erro YOLO"] = (w[M_YOLO] - w["real"]).abs()
piores_cls = w.nlargest(10, "erro clássico")[["imagem", "real", M_CLS, M_YOLO, "erro clássico", "erro YOLO"]]
print("a) washers_hardee — as 10 piores imagens do clássico (pilhas densas, outras peças não rotuladas):")
display(piores_cls)
print(f"   erro médio nessas imagens: clássico {piores_cls['erro clássico'].mean():.2f} "
      f"× YOLO {piores_cls['erro YOLO'].mean():.2f}")
piores_cls.to_csv(COMPARACAO / f"hipotese_a_washers_{SUFIXO}.csv", index=False)

print("\nb) screws_and_nuts, porca (porcas enroscadas em parafusos):")
chave_b = ("classe_dataset", "porca", "screws_and_nuts")
if chave_b in comparacao.index:
    display(comparacao.loc[[chave_b], ["n", "MAE clássico", "MAE YOLO", "viés clássico", "viés YOLO"]])
else:
    print("   (sem imagens desse recorte após as exclusões)")

### Célula 7 — As 10 piores imagens do YOLO

Desenha as 10 imagens em que o YOLO mais errou no test, com as caixas previstas (cor da classe) e as do GT (magenta). Refaz a inferência para conferir se bate com a previsão congelada. Como a inferência aqui é feita imagem por imagem, detecções com confiança muito perto do limiar podem mudar de lado em relação à execução em lote.


In [ ]:
# Célula 7 — As 10 piores imagens do YOLO, com as caixas previstas e as do GT
pasta = COMPARACAO / f"piores_yolo_{SUFIXO}"
pasta.mkdir(parents=True, exist_ok=True)
piores = (df.assign(erro_yolo=(df[M_YOLO] - df["real"]).abs())
            .sort_values(["erro_yolo", "real"], ascending=False).head(10))


def caixas_gt(caminho_label, largura, altura):
    """Lê um label YOLO e devolve as caixas do GT em pixels (x1, y1, x2, y2)."""
    if not caminho_label.exists() or caminho_label.stat().st_size == 0:
        return np.zeros((0, 4))
    v = np.loadtxt(caminho_label, ndmin=2)[:, 1:5]
    xc, yc = v[:, 0] * largura, v[:, 1] * altura
    lw, lh = v[:, 2] * largura, v[:, 3] * altura
    return np.stack([xc - lw / 2, yc - lh / 2, xc + lw / 2, yc + lh / 2], axis=1)


divergencias = []
fig, eixos = plt.subplots(5, 2, figsize=(14, 32))
for ax, (_, r) in zip(eixos.ravel(), piores.iterrows()):
    pasta_classe = DATA / "yolo" / r["classe"] / "test"
    caminho = pasta_classe / "images" / r["imagem"]
    bgr = cv2.imread(str(caminho))
    assert bgr is not None, f"Imagem não encontrada: {caminho}"
    prev, _ = detectar(modelos[r["classe"]], str(caminho), LIMIARES[r["classe"]])
    if len(prev) != r[M_YOLO]:
        divergencias.append((r["imagem"], r["classe"], len(prev), r[M_YOLO]))
    esp = max(1, round(max(bgr.shape[:2]) / 400))
    gt_cx = caixas_gt(pasta_classe / "labels" / (Path(r["imagem"]).stem + ".txt"), bgr.shape[1], bgr.shape[0])
    desenhar(bgr, gt_cx, COR_GT_BGR, espessura=esp)
    desenhar(bgr, prev, CORES_BGR[r["classe"]], espessura=esp + 1)
    cv2.imwrite(str(pasta / f"{Path(r['imagem']).stem}_{r['classe']}.jpg"), bgr)
    ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    ax.axis("off")
    ax.set_title(f"{r['dataset']} | {r['classe']} | real {r['real']} × YOLO {r[M_YOLO]}", fontsize=10)
plt.suptitle("Magenta = GT | cor da classe = previsão do YOLO", y=1.0)
plt.tight_layout()
plt.show()
piores.to_csv(COMPARACAO / f"piores_yolo_{SUFIXO}.csv", index=False)

if divergencias:
    print("ATENÇÃO: a contagem refeita difere da previsão congelada (imagem, classe, agora, congelada):")
    for d in divergencias:
        print("  ", d)
else:
    print("Contagens refeitas idênticas às previsões congeladas.")

### Célula 8 — Gráficos para o relatório

Gera o gráfico de MAE por classe e o de contagem prevista contra a real nos dois métodos, e salva os PNG em `saidas/comparacao/`.


In [ ]:
# Célula 8 — Gráficos para o relatório
por_classe = comparacao.xs("classe", level="escopo").droplevel("dataset")
x = np.arange(len(por_classe))
larg = 0.38
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - larg / 2, por_classe["MAE clássico"], larg, label="Clássico")
ax.bar(x + larg / 2, por_classe["MAE YOLO"], larg, label="YOLO")
ax.set_xticks(x, por_classe.index)
ax.set_ylabel("MAE (objetos por imagem)")
ax.set_title("Erro médio de contagem no test")
ax.legend()
plt.tight_layout()
plt.savefig(COMPARACAO / f"mae_por_classe_{SUFIXO}.png", dpi=150)
plt.show()

lim = max(df["real"].max(), df[METODOS].max().max()) * 1.05
fig, eixos = plt.subplots(1, 2, figsize=(12, 5.5), sharex=True, sharey=True)
for ax, m, rotulo in zip(eixos, METODOS, ["Clássico", "YOLO"]):
    for c in CLASSES:
        g = df[df["classe"] == c]
        ax.scatter(g["real"], g[m], s=18, alpha=0.7, label=c,
                   color=tuple(v / 255 for v in CORES_BGR[c][::-1]))
    ax.plot([0, lim], [0, lim], "k--", lw=1)
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_title(rotulo)
    ax.set_xlabel("Contagem real")
eixos[0].set_ylabel("Contagem prevista")
eixos[0].legend()
plt.tight_layout()
plt.savefig(COMPARACAO / f"previsto_vs_real_{SUFIXO}.png", dpi=150)
plt.show()
print(f"Tabelas e figuras salvas em {COMPARACAO}")

### Célula 9 — Demonstração: contar peças em qualquer imagem

Cole o link direto de uma imagem em `URL_IMAGEM` (clique com o botão direito na imagem e escolha *Copiar endereço da imagem*), ou mude `ENVIAR_DO_COMPUTADOR` para `True` e escolha um arquivo. Os três detectores rodam sobre a imagem e o resultado aparece desenhado, com a contagem por classe.

Cada classe tem um detector independente, então a mesma peça pode ser detectada em duas classes (por exemplo, uma arruela também como porca). A demonstração resolve isso mantendo só a caixa de maior confiança quando caixas de classes diferentes se sobrepõem com IoU maior ou igual a 0,6. Essa regra existe só aqui e não faz parte dos resultados oficiais do test, que foram gerados sem ela. A célula também imprime a contagem sem a resolução, para comparação.

Limitações para interpretar o resultado:
- Os detectores funcionam melhor em imagens parecidas com as de treino: peças metálicas, vistas de cima, sobre fundo liso.
- A resolução de conflitos reduz a contagem duplicada, mas não a elimina. Quando só uma das classes detecta a peça (por exemplo, uma arruela com confiança abaixo do limiar de 0,80 no detector de arruela, detectada apenas como porca), não há conflito a resolver e a peça é contada na classe errada.
- Os limiares são os congelados na etapa 4. Esta demonstração não faz parte da avaliação.


In [ ]:
# Célula 9 — Demonstração: contar peças em qualquer imagem
URL_IMAGEM = ""                # link direto de uma imagem (.jpg, .png, .webp)
ENVIAR_DO_COMPUTADOR = False   # True para escolher um arquivo do computador

import requests
from PIL import Image


def iou(a, b):
    """IoU entre todas as caixas de a e de b (formato x1, y1, x2, y2)."""
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda c: (c[:, 2] - c[:, 0]) * (c[:, 3] - c[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter)


def resolver_conflitos(det, limiar_iou=0.6):
    """Em caixas de classes diferentes muito sobrepostas, mantém só a de maior confiança."""
    manter = {c: np.ones(len(det[c][1]), bool) for c in det}
    for i, ci in enumerate(CLASSES):
        for cj in CLASSES[i + 1:]:
            (bi, si), (bj, sj) = det[ci], det[cj]
            if len(si) == 0 or len(sj) == 0:
                continue
            for a, b in zip(*np.nonzero(iou(bi, bj) >= limiar_iou)):
                if si[a] >= sj[b]:
                    manter[cj][b] = False
                else:
                    manter[ci][a] = False
    return {c: (det[c][0][manter[c]], det[c][1][manter[c]]) for c in det}

if ENVIAR_DO_COMPUTADOR:
    from google.colab import files
    enviado = files.upload()
    assert enviado, "Nenhum arquivo enviado."
    conteudo = next(iter(enviado.values()))
else:
    assert URL_IMAGEM, "Preencha URL_IMAGEM ou use ENVIAR_DO_COMPUTADOR = True."
    resp = requests.get(URL_IMAGEM, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    resp.raise_for_status()
    conteudo = resp.content

try:
    imagem = Image.open(io.BytesIO(conteudo)).convert("RGB")
except Exception as erro:
    raise ValueError("O conteúdo não é uma imagem. Use o link direto do arquivo "
                     "(botão direito na imagem → Copiar endereço da imagem).") from erro

bgr = cv2.cvtColor(np.array(imagem), cv2.COLOR_RGB2BGR)
det = {c: detectar(modelos[c], bgr, LIMIARES[c]) for c in CLASSES}
det_final = resolver_conflitos(det)
brutas = {c: len(det[c][1]) for c in CLASSES}
contagens = {c: len(det_final[c][1]) for c in CLASSES}
desenho = bgr.copy()
for c in CLASSES:
    caixas, confs = det_final[c]
    desenhar(desenho, caixas, CORES_BGR[c], [f"{c} {p:.2f}" for p in confs])
print("sem resolver conflitos:", brutas)

plt.figure(figsize=(12, 9))
plt.imshow(cv2.cvtColor(desenho, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(" | ".join(f"{c}: {n}" for c, n in contagens.items()) + f" | total: {sum(contagens.values())}")
plt.show()
print(contagens)